### RouteHunter — Build Notebook

RouteHunter build

In [1]:
import os
import csv
import shutil

### 0. Create app folder

In [2]:
DATA_DIR = "tmp"
STATIC_DIR = os.path.join(DATA_DIR, "static")   
MODEL_DIR = os.path.join(DATA_DIR, "model")    
BUILD_DIR = os.path.join(DATA_DIR, "build")

# create folders
for path in [STATIC_DIR, MODEL_DIR, BUILD_DIR]:
    os.makedirs(path, exist_ok=True)
    print(f"Ready: {path}")

Ready: tmp/static
Ready: tmp/model
Ready: tmp/build


In [3]:
# Files the app reads by name -- these become the config manifest, one row each.
PATHS = {
    "seed": os.path.join(STATIC_DIR, "seed.csv"),
    "casp": os.path.join(STATIC_DIR, "casp.csv"),
    "monitor": os.path.join(STATIC_DIR, "monitor.csv"),
    "candidate": os.path.join(STATIC_DIR, "candidate.csv"),
    "abstract": os.path.join(STATIC_DIR, "abstract.csv"),
    "aizynthfinder": os.path.join(MODEL_DIR, "aizynthfinder.pickle"),
    "synplanner": os.path.join(MODEL_DIR, "synplanner.pickle"),
    "paper": os.path.join(MODEL_DIR, "paper.pickle"),
}
COMMENTS = {
    "seed": "Digitalized collection of targets",
    "casp": "CASP solved-by-tool table",
    "monitor": "High-confidence paper with route candidates",
    "candidate": "Medium-confidence paper with route candidates",
    "abstract": "Training data for paper classifier",
    "aizynthfinder": "AiZynthFinder solvability model",
    "synplanner": "SynPlanner solvability model",
    "paper": "Paper with route classifier model",
}

with open(os.path.join(DATA_DIR, "config.csv"), "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["key", "path", "comment"])
    writer.writeheader()
    for key, path in PATHS.items():
        writer.writerow({"key": key, "path": os.path.relpath(path, DATA_DIR), "comment": COMMENTS[key]})
print(f"Wrote {DATA_DIR}/config.csv ({len(PATHS)} entries)")

Wrote tmp/config.csv (8 entries)


In [4]:
# Build-only: nothing outside this notebook reads these by name.
PROCESSED_PAPERS = os.path.join(BUILD_DIR, "processed.csv")
CANDIDATE_PAPERS = "extraction_data/openalex_chem_metadata_1.csv"
AIZYNTHFINDER_DATA = os.path.join(BUILD_DIR, "aizynthfinder.csv")
SYNPLANNER_DATA = os.path.join(BUILD_DIR, "synplanner.csv")

### 1. Collect and validate targets

In [5]:
import pandas as pd
from rdkit import Chem

In [6]:
# processed data
SHEET_ID_OPRD = "18aP203JdmhgdD67P-vTjparpQqh90lN80RjOaIGSyKs"
SHEET_ID_OTHER = "1p-n7Y03KD8fkRSwpkFf5hNBKzr4Kj3cMyh6YpLulnXE"
URL_OPRD = f"https://docs.google.com/spreadsheets/d/{SHEET_ID_OPRD}/export?format=csv"
URL_OTHER = f"https://docs.google.com/spreadsheets/d/{SHEET_ID_OTHER}/export?format=csv"

In [7]:
sheets_df = pd.concat([pd.read_csv(URL_OPRD), pd.read_csv(URL_OTHER)])
processed_df = sheets_df.dropna()
collected_df = processed_df[processed_df["target"] != "SKIP"]

print(f"Total papers:      {len(sheets_df)}")
print(f"Processed papers:  {len(processed_df)} ({100 * len(processed_df) / len(sheets_df):.1f}%)")
print(f"Collected targets: {len(collected_df)}")

Total papers:      4310
Processed papers:  2587 (60.0%)
Collected targets: 1459


In [8]:
for _, row in collected_df.iterrows():
    
    # check smiles
    mol = Chem.MolFromSmiles(row["target"])
    if mol is None:
        print(row["doi"], row["target"])
        raise("Unparseable SMILES")

    # check inchikey
    inchikey = Chem.MolToInchiKey(mol)
    if not inchikey:
        print(row["doi"], row["target"])
        raise("InChIKey generation failed")
#
print(f"Checked {len(collected_df)} targets")

Checked 1459 targets


### 2. Fetch paper metadata

In [9]:
import pandas as pd

In [10]:
candidates_df = pd.read_csv(CANDIDATE_PAPERS).set_index("doi")

In [11]:
res = []
for _, paper in processed_df.iterrows():
    
    if paper["doi"] not in candidates_df.index:
        print(f"DOI not found among candidate papers: {paper['doi']!r}")
        continue
        
    row = candidates_df.loc[paper["doi"]]
    res.append({
        "journal": row["journal"],
        "title": row["title"],
        "abstract": row["abstract"],
        "year": pd.to_datetime(row["publication_date"]).year,
        "doi": paper["doi"],
        "target": paper["target"],
        "contributor": "Dmitry Zankov",
        "has_route": 0 if paper["target"] == "SKIP" else 1,
    })
#
processed_df = pd.DataFrame(res)
print(f"Matched data for {len(res)} papers")

DOI not found among candidate papers: '10.1021/acs.oprd.5c00448'
DOI not found among candidate papers: '10.1021/acs.oprd.5c00443'
Matched data for 2585 papers


### 3. Store static datasets

### 3.1 Store processed papers

In [12]:
processed_df.to_csv(PROCESSED_PAPERS, index=False)
print(f"Saved processed papers ({len(processed_df)} rows)")

Saved processed papers (2585 rows)


### 3.2 Store target papers

In [13]:
processed_df = pd.read_csv(PROCESSED_PAPERS)
seed_df = processed_df[processed_df["has_route"] == 1].drop(columns="has_route")
seed_df.to_csv(PATHS["seed"], index=False)
print(f"Saved seed dataset ({len(seed_df)} targets)")

Saved seed dataset (1457 targets)


### 4. CASP data and models

In [14]:
from routehunter_build.models import load_solvability_data, train_solvability_model, save_model
from routehunter_build.merge import merge_tool_tables

### 4.1 Train AiZynthFinder model

In [15]:
az_data = pd.read_json("../oprd/aizynth_oprd.json", orient="table")[["target", "is_solved"]]
az_data.columns = ["smiles", "is_solved"]
az_data.to_csv(AIZYNTHFINDER_DATA, index=False)

print(f"Saved AiZynthFinder training data ({len(az_data)} molecules, {az_data['is_solved'].sum()} solved)")

Saved AiZynthFinder training data (1152 molecules, 727 solved)


In [16]:
smiles, y = load_solvability_data(AIZYNTHFINDER_DATA)
print(f"Training on {len(smiles)} molecules ({sum(y)} solved, {len(y) - sum(y)} not solved)")

model, meta = train_solvability_model(smiles, y, random_state=42)
print(f"Model metrics (5x5-CV): precision={meta['precision']:.3f}  recall={meta['recall']:.3f}  f1={meta['f1']:.3f}")

save_model(model, PATHS["aizynthfinder"])
print("Saved AiZynthFinder model")

Training on 1152 molecules (727 solved, 425 not solved)
Model metrics (5x5-CV): precision=0.801  recall=0.862  f1=0.830
Saved AiZynthFinder model


### 4.2 Train SynPlanner model

In [17]:
sp_data = pd.read_csv("../oprd/synplan_oprd/tree_search_stats.csv")[["target_smiles", "solved"]]
sp_data.columns = ["smiles", "is_solved"]
sp_data.to_csv(SYNPLANNER_DATA, index=False)

print(f"Saved SynPlanner training data ({len(sp_data)} molecules, {sp_data['is_solved'].sum()} solved)")

Saved SynPlanner training data (1136 molecules, 763 solved)


In [18]:
smiles, y = load_solvability_data(SYNPLANNER_DATA)
print(f"Training on {len(smiles)} molecules ({sum(y)} solved, {len(y) - sum(y)} not solved)")

model, meta = train_solvability_model(smiles, y, random_state=42)
print(f"Model metrics (5x5-CV): precision={meta['precision']:.3f}  recall={meta['recall']:.3f}  f1={meta['f1']:.3f}")

save_model(model, PATHS["synplanner"])
print("Saved SynPlanner model")

Training on 1136 molecules (763 solved, 373 not solved)
Model metrics (5x5-CV): precision=0.801  recall=0.817  f1=0.809
Saved SynPlanner model


### 4.3 Merged casp table

In [19]:
casp_df = merge_tool_tables({"AiZynthFinder": AIZYNTHFINDER_DATA, "SynPlanner": SYNPLANNER_DATA})
casp_df.to_csv(PATHS["casp"], index=False)

print(f"Saved merged solvability table ({len(casp_df)} molecules)")

[14:33:54] Invalid InChI prefix in generating InChI Key


Saved merged solvability table (1057 molecules)


## 5. Paper-with-route probability model

In [20]:
import pandas as pd
from routehunter_build.models import load_abstract_data, train_abstract_model, save_model
from routehunter_build.models import find_threshold_for_precision
from routehunter_build.models import combine_text, load_model

### 5.1 Title/abstract training data

In [21]:
processed_df = pd.read_csv(PROCESSED_PAPERS)

abstract_df = processed_df.drop_duplicates(subset="doi")
abstract_df = abstract_df[["title", "abstract", "doi", "has_route"]].dropna(subset="title")
abstract_df.to_csv(PATHS["abstract"], index=False)

print(f"Saved abstract training data ({len(abstract_df)} papers, {abstract_df['has_route'].sum()} with a route)")

Saved abstract training data (2368 papers, 1240 with a route)


### 5.2 Train paper classifier

In [22]:
text, y = load_abstract_data(PATHS["abstract"])
print(f"Training on {len(text)} labeled papers ({sum(y)} with a route, {len(y) - sum(y)} without)")

model, meta = train_abstract_model(text, y, random_state=42)
print(f"Model metrics (5x5-CV): precision={meta['precision']:.3f}  recall={meta['recall']:.3f}  f1={meta['f1']:.3f}")

save_model(model, PATHS["paper"])
print("Saved the paper classifier")

Training on 2368 labeled papers (1240 with a route, 1128 without)
Model metrics (5x5-CV): precision=0.827  recall=0.843  f1=0.835
Saved the paper classifier


### 5.3 Calibrate probability threshold

In [23]:
PRECISION_HIGH = 0.9
PRECISION_MEDIUM = 0.8

high = find_threshold_for_precision(meta["y_val"], meta["y_prob"], PRECISION_HIGH)
medium = find_threshold_for_precision(meta["y_val"], meta["y_prob"], PRECISION_MEDIUM)

print(f"High threshold:   {high.threshold:.3f}  (precision={high.precision:.3f}, recall={high.recall:.3f})")
print(f"Medium threshold: {medium.threshold:.3f}  (precision={medium.precision:.3f}, recall={medium.recall:.3f})")

threshold_high = high.threshold
threshold_medium = medium.threshold

High threshold:   0.734  (precision=0.900, recall=0.523)
Medium threshold: 0.428  (precision=0.800, recall=0.905)


### 5.4 Score candidate papers

In [24]:
model = load_model(PATHS["paper"])

candidates_df = pd.read_csv(CANDIDATE_PAPERS)
print(f"Loaded {len(candidates_df)} candidate papers")

candidate_text = [combine_text(t, a) for t, a in zip(candidates_df["title"], candidates_df["abstract"])]
candidates_df["route_prob"] = model.predict_proba(candidate_text)[:, 1]

candidates_df = candidates_df[["journal", "title", "abstract", "doi", "publication_date", "route_prob"]]
print(f"Scored {len(candidates_df)} candidate papers")

Loaded 654806 candidate papers
Scored 654806 candidate papers


### 5.5 High-confidence papers (monitor)

In [27]:
high_df = candidates_df[candidates_df["route_prob"] >= threshold_high].sort_values("route_prob", ascending=False)
high_df.to_csv(PATHS["monitor"], index=False)

print(f"Saved high-confidence papers ({len(high_df)} papers, route_prob >= {threshold_high:.3f})")

Saved high-confidence papers (13432 papers, route_prob >= 0.734)


### 5.6 Medium-confidence paper (candidate)

In [28]:
medium_df = candidates_df[candidates_df["route_prob"] >= threshold_medium].sort_values("route_prob", ascending=False)
medium_df.to_csv(PATHS["candidate"], index=False)

print(f"Saved medium-confidence papers ({len(medium_df)} papers, route_prob >= {threshold_medium:.3f}, "
      f"includes the {len(high_df)} high-tier papers)")

Saved medium-confidence papers (127801 papers, route_prob >= 0.428, includes the 13432 high-tier papers)
